# Анализируем данные для агентства недвижимости в SQL

* Автор: Трутнева Александра
* Дата: 13.01.2026

Необходим анализ объявлений о продаже жилой недвижимости в Санкт-Петербурге и Ленинградской области, чтобы найти самые перспективные сегменты недвижимости. В нашем распоряжении архивная информация сервиса недвижимости (в целях конфиденциальности название не приводится): объявления о продаже жилой недвижимости в Санкт-Петербурге и Ленинградской области за несколько лет.

## Цели и задачи проекта

<font color='#777778'>
Цель: глубже понять поведение пользователей и найти примечательные закономерности.<br>

Задачи:
- Познакомиться с данными, сосредоточившись на тех, что помогут решить задачи проекта с помощью SQL и интерпретировать полученные результаты.
- Решить ad hoc задачи с помощью SQL-запросов, чтобы помочь заказчику спланировать бизнес-стратегию, выбрать сегмент недвижимости с высоким спросом и подходящий сезон для публикации объявлений:
    - определить наиболее привлекательные сегменты недвижимости Санкт-Петербурга и городов Ленинградской области, чтобы создать эффективную бизнес-стратегию на рынке недвижимости (по времени активности объявлений);
    - понять сезонные тенденции на рынке Санкт-Петербурга и городов Ленинградской области, чтобы знать периоды с повышенной активностью продавцов и покупателей недвижимости в регионе и спланировать маркетинговые кампании, выбрав сроки для выхода на рынок.
</font>

## Содержимое проекта

[Знакомство с данными](#metka_1)

[Решение ad hoc задачи 1 (Время активности объявлений)](#metka_2)
    
[Решение ad hoc задачи 2 (Сезонность объявлений)](#metka_3)

[Выводы](#metka_4)

[Рекомендации](#metka_5)

---

<a id='metka_1'></a>
### Знакомство с данными

```sql
--Определение периода, за который представлены объявления о продаже недвижимости
SELECT
--определение начала периода
MIN(first_day_exposition)
--определение конца периода
, MAX(first_day_exposition)
FROM advertisement;

--Результат
--min       |max       |
------------+----------+
--2014-11-27|2019-05-03|


--Определение для каждого типа населённого пункта количества населённых пунктов и количества объявлений
SELECT
--вывод сгруппированных типов населённых пунктов
t.TYPE
--определение количества населённых пунктов в разрезе их типов (в т.ч. Санкт-Петербург)
, COUNT(DISTINCT c.city_id) AS count_type_id
--определение количества объявлений в разрезе типов населённых пунктов
, COUNT(a.id) AS count_id
FROM type AS t
LEFT JOIN flats AS f ON t.type_id=f.type_id
LEFT JOIN advertisement AS a ON f.id=a.id
LEFT JOIN city AS c ON c.city_id=f.city_id
--группировка по типам населённых пунктов
GROUP BY t.TYPE
--сортировка по количеству населённых пунктов в разрезе их типов в порядке убывания
ORDER BY count_type_id DESC;

--Результат
--type                                     |count_type_id|count_id|
-------------------------------------------+-------------+--------+
--посёлок                                  |          113|    2092|
--деревня                                  |          106|     945|
--город                                    |           43|   20008|
--посёлок городского типа                  |           30|     363|
--городской посёлок                        |           13|     187|
--село                                     |            9|      32|
--посёлок при железнодорожной станции      |            6|      15|
--садовое товарищество                     |            4|       4|
--коттеджный посёлок                       |            3|       3|
--садоводческое некоммерческое товарищество|            1|       1|


--Определение основных статистик по полю со временем активности объявлений
SELECT
--определение минимального значения
MIN(days_exposition) AS day_min
--определение максимального значения
, MAX(days_exposition) AS day_max
--определение среднего значения, округленного до двух знаков после запятой
, ROUND(AVG(days_exposition)::numeric,2) AS day_avg
--определение медианы
, PERCENTILE_DISC(0.5) WITHIN GROUP(ORDER BY days_exposition) AS day_median
FROM advertisement;

--Результат
--day_min|day_max|day_avg|day_median|
---------+-------+-------+----------+
--    1.0| 1580.0| 180.75|      95.0|


--Определение процента объявлений, которые сняли с публикации, округлённого до двух знаков после запятой 
SELECT
ROUND(COUNT(days_exposition)::numeric/COUNT(first_day_exposition)::numeric*100,2)
FROM advertisement;

--Результат
--round|
-------+
--86.55|


--Определение процента объявлений о продаже квартир в Санкт-Петербурге, округлённого до двух знаков после запятой
SELECT
--округление до двух знаков после запятой доли объявлений по Санкт-Петербургу (рассчитанной подзапросом) от общего числа объявлений, переведённой в проценты
ROUND((SELECT
--подсчёт объявлений, удовлетворяющих фильтру
COUNT(a.id) AS cls_count
--объединение необходимых таблиц по их первичным ключам
FROM advertisement AS a
LEFT JOIN flats AS f ON a.id=f.id
LEFT JOIN city AS c ON c.city_id=f.city_id
--установка фильтра для рассмотрения только Санкт-Петербурга
WHERE c.city='Санкт-Петербург')::numeric/COUNT(id)*100,2)
FROM advertisement;

--Результат
--round|
-------+
--66.47|


--Определение основных статистических показателей для значений стоимости одного квадратного метра, округлённых до двух знаков после запятой
SELECT
--определение минимальной стоимости недвижимости за квадратный метр
	ROUND(MIN(a.last_price::numeric/f.total_area::numeric),2) AS price_min
--определение максимальной стоимости недвижимости за квадратный метр
	, ROUND(MAX(a.last_price::numeric/f.total_area::numeric),2) AS price_max
--определение средней стоимости недвижимости за квадратный метр
	, ROUND(AVG(a.last_price::numeric/f.total_area::numeric),2) AS price_avg
--определение медианы стоимости недвижимости за квадратный метр
	, ROUND(PERCENTILE_DISC(0.5) WITHIN GROUP(ORDER BY a.last_price::numeric/f.total_area::numeric),2) AS price_median
--объединение необходимых таблиц по их первичным ключам
FROM advertisement AS a
LEFT JOIN flats AS f ON a.id=f.id;

--Результат
--price_min|price_max |price_avg|price_median|
-----------+----------+---------+------------+
--   111.83|1907500.00| 99432.25|    95000.00|


--Проверка корректности данных и определение статистических показателей
SELECT
--определение минимальных значений:
--минимальное значение общей площади недвижимости
	MIN(total_area) AS total_area_min
--минимальное значение количества комнат
	, MIN(rooms) AS rooms_min
--минимальное значение количества балконов
	, MIN(balcony) AS balcony_min
--минимальное значение высоты потолков
	, MIN(ceiling_height) AS ceiling_height_min
--минимальное значение этажа
	, MIN(floor) AS floor_min
--определение максимальных значений:
--максимальное значение общей площади недвижимости
	, MAX(total_area) AS total_area_max
--максимальное значение количества комнат
	, MAX(rooms) AS rooms_max
--максимальное значение количества балконов
	, MAX(balcony) AS balcony_max
--максимальное значение высоты потолков
	, MAX(ceiling_height) AS ceiling_height_max
--максимальное значение этажа
	, MAX(floor) AS floor_max
--определение средних значений:
--среднее значение общей площади недвижимости
	, ROUND(AVG(total_area)::numeric,2) AS total_area_avg
--среднее значение количества комнат
	, ROUND(AVG(rooms)::numeric,2) AS rooms_avg
--среднее значение количества балконов
	, ROUND(AVG(balcony)::numeric,2) AS balcony_avg
--среднее значение высоты потолков
	, ROUND(AVG(ceiling_height)::numeric,2) AS ceiling_height_avg
--среднее значение этажа
	, ROUND(AVG(floor)::numeric,2) AS floor_avg
--определение медианных значений:
--медианное значение общей площади недвижимости
	, PERCENTILE_DISC(0.5) WITHIN GROUP(ORDER BY total_area) AS total_area_median
--медианное значение количества комнат
	, PERCENTILE_DISC(0.5) WITHIN GROUP(ORDER BY rooms) AS rooms_median
--медианное значение количества балконов
	, PERCENTILE_DISC(0.5) WITHIN GROUP(ORDER BY balcony) AS balcony_median
--медианное значение высоты потолков
	, PERCENTILE_DISC(0.5) WITHIN GROUP(ORDER BY ceiling_height) AS ceiling_height_median
--медианное значение этажа
	, PERCENTILE_DISC(0.5) WITHIN GROUP(ORDER BY floor) AS floor_median
--определение 99 перцентиля:
--99 перцентиль общей площади недвижимости
	, PERCENTILE_DISC(0.99) WITHIN GROUP(ORDER BY total_area) AS total_area_99perc
--99 перцентиль значение количества комнат
	, PERCENTILE_DISC(0.99) WITHIN GROUP(ORDER BY rooms) AS rooms_99perc
--99 перцентиль количества балконов
	, PERCENTILE_DISC(0.99) WITHIN GROUP(ORDER BY balcony) AS balcony_99perc
--99 перцентиль высоты потолков
	, PERCENTILE_DISC(0.99) WITHIN GROUP(ORDER BY ceiling_height) AS ceiling_height_99perc
--99 перцентиль этажа
	, PERCENTILE_DISC(0.99) WITHIN GROUP(ORDER BY floor) AS floor_99perc
FROM flats;

--Результат
--total_area_min|rooms_min|balcony_min|ceiling_height_min|floor_min|total_area_max|rooms_max|balcony_max|ceiling_height_max|
----------------+---------+-----------+------------------+---------+--------------+---------+-----------+------------------+
--          12.0|        0|        0.0|               1.0|        1|         900.0|       19|        5.0|             100.0|


--floor_max|total_area_avg|rooms_avg|balcony_avg|ceiling_height_avg|floor_avg|total_area_median|rooms_median|balcony_median|
-----------+--------------+---------+-----------+------------------+---------+-----------------+------------+--------------+
--       33|         60.33|     2.07|       1.15|              2.77|     5.89|             52.0|           2|           1.0|


--ceiling_height_median|floor_median|total_area_99perc|rooms_99perc|balcony_99perc|ceiling_height_99perc|floor_99perc|
-----------------------+------------+-----------------+------------+--------------+---------------------+------------+
--                 2.65|           4|            197.9|           5|           5.0|                 3.83|          23|
```

<a id='metka_2'></a>
### Решение ad hoc задачи 1 (Время активности объявлений)

```sql
-- Задача 1: Время активности объявлений
-- CTE, определяющий аномальные значения (выбросы) по значению перцентилей
WITH limits AS (
    SELECT  
        PERCENTILE_DISC(0.99) WITHIN GROUP (ORDER BY total_area) AS total_area_limit,
        PERCENTILE_DISC(0.99) WITHIN GROUP (ORDER BY rooms) AS rooms_limit,
        PERCENTILE_DISC(0.99) WITHIN GROUP (ORDER BY balcony) AS balcony_limit,
        PERCENTILE_DISC(0.99) WITHIN GROUP (ORDER BY ceiling_height) AS ceiling_height_limit_h,
        PERCENTILE_DISC(0.01) WITHIN GROUP (ORDER BY ceiling_height) AS ceiling_height_limit_l
    FROM flats     
),

-- CTE, определяющий id объявлений, которые не содержат выбросы, оставляющий пропущенные данные
filtered_id AS(
    SELECT id
    FROM flats  
    WHERE 
        total_area < (SELECT total_area_limit FROM limits)
        AND (rooms < (SELECT rooms_limit FROM limits) OR rooms IS NULL)
        AND (balcony < (SELECT balcony_limit FROM limits) OR balcony IS NULL)
        AND ((ceiling_height < (SELECT ceiling_height_limit_h FROM limits)
            AND ceiling_height > (SELECT ceiling_height_limit_l FROM limits)) OR ceiling_height IS NULL)
),

-- CTE, раздедяющий объявления на категории
category_base AS(
	SELECT
		f.id
		, a.first_day_exposition
		, t.type AS t
		, f.total_area
		, f.rooms
		, f.balcony
		, f.floor
		, f.ceiling_height
		, f.open_plan
--определение стоимости квадратного метра недвижимости 
		, a.last_price::numeric/f.total_area::numeric AS sold_price
--разделение объявлений на категории по принадлежности объявления к населённым пунктам Ленинградской области или Санкт-Петербургу
		, CASE
			WHEN city_id='6X8I' THEN 'Санкт-Петербург'
			ELSE 'ЛенОбл'
		END AS region
--разделение объявлений на категории по количеству дней активности
		, CASE
			WHEN days_exposition<31 THEN 'до 1 месяца'
			WHEN days_exposition BETWEEN 31 AND 90 THEN 'до 3 месяцев'
		 	WHEN days_exposition BETWEEN 91 AND 180 THEN 'до 6 месяцев'
			WHEN days_exposition>181 THEN 'свыше 6 месяцев'
			ELSE 'non category'
		END AS activ
--разделение объявлений на категории по признаку является ли квартира студией или нет
		, CASE
			WHEN kitchen_area IS NULL AND rooms BETWEEN 0 AND 1 THEN 'студия'
			ELSE 'не студия'
		END AS studio
--объединение необходимых таблиц по их первичным ключам
	FROM flats AS f
	LEFT JOIN advertisement AS a ON f.id=a.id
	LEFT JOIN type AS t ON f.type_id=t.type_id
)

-- Основной запрос, использующий id объявлений (СТЕ filtered_id), которые не содержат выбросы при анализе данных
SELECT
--вывод данных, сгруппированных в разрезе регион-тип населённого пункта-сегмент активности-студийность
	region
	, activ
--определение числа проданных квартир
	, COUNT(id) AS sold_count
--определение доли проданных квартир от общего числа проданных квартир
	, COUNT(id)::numeric/(SELECT COUNT(id) FROM category_base WHERE id IN (SELECT * FROM filtered_id) AND first_day_exposition BETWEEN '2015-01-01' AND '2018-12-31') AS sold_part
--определение средней стоимости квадратного метра недвижимости
	, ROUND(AVG(sold_price),2) AS sold_price_avg
--определение средней площади недвижимости
	, ROUND(AVG(total_area)::numeric,2) AS total_area_avg
--определение медианы количества комнат
	, PERCENTILE_DISC(0.5) WITHIN GROUP(ORDER BY rooms) AS rooms_median
--определение медианы количества балконов
	, PERCENTILE_DISC(0.5) WITHIN GROUP(ORDER BY balcony) AS balcony_median
--определение медианы этажности
	, PERCENTILE_DISC(0.5) WITHIN GROUP(ORDER BY floor) AS floor_median
--определение средней высоты потолка
	, ROUND(AVG(ceiling_height)::numeric,2) AS ceiling_height_avg
FROM category_base
--фильтрация с объявлениями, опубликованными в период с 2015 по 2018 годы, исключающая объявления с аномальными значениями
WHERE t='город' AND id IN (SELECT * FROM filtered_id) AND first_day_exposition BETWEEN '2015-01-01' AND '2018-12-31'
--группировка данных в разрезе регион-тип населённого пункта-сегмент активности-студийность
GROUP BY
	region
	, t
	, activ
ORDER BY
	region DESC
	, activ;

--Результат
--region         |activ          |sold_count|sold_part             |sold_price_avg|total_area_avg|rooms_median|balcony_median|
-----------------+---------------+----------+----------------------+--------------+--------------+------------+--------------+
--Санкт-Петербург|non category   |       670|0.03970135103105001185|     135732.84|         81.10|           3|           1.0|
--Санкт-Петербург|до 1 месяца    |      1794|0.10630481156672197203|     108919.78|         54.66|           2|           1.0|
--Санкт-Петербург|до 3 месяцев   |      3020|0.17895235837876273999|     110874.32|         56.58|           2|           1.0|
--Санкт-Петербург|до 6 месяцев   |      2244|0.13296989808011377104|     111973.67|         60.55|           2|           1.0|
--Санкт-Петербург|свыше 6 месяцев|      3489|0.20674330410049774828|     114950.11|         65.74|           2|           1.0|
--ЛенОбл         |non category   |       209|0.01238445129177530220|      73037.82|         62.85|           2|           1.0|
--ЛенОбл         |до 1 месяца    |       340|0.02014695425456269258|      71907.63|         48.75|           2|           1.0|
--ЛенОбл         |до 3 месяцев   |       864|0.05119696610571225409|      67423.80|         50.85|           2|           1.0|
--ЛенОбл         |до 6 месяцев   |       553|0.03276842853756814411|      69809.30|         51.83|           2|           1.0|
--ЛенОбл         |свыше 6 месяцев|       862|0.05107845461009717943|      68127.85|         54.91|           2|           1.0|



--floor_median|ceiling_height_avg|
--------------+------------------+
--           4|              2.90|
--           5|              2.76|
--           5|              2.77|
--           5|              2.79|
--           5|              2.83|
--           3|              2.79|
--           4|              2.70|
--           3|              2.71|
--           3|              2.70|
--           3|              2.72|

```

<a id='metka_3'></a>
### Решение ad hoc задачи 2 (Сезонность объявлений)

```sql
-- Задача 2: Сезонность объявлений
-- CTE, определяющий аномальные значения (выбросы) по значению перцентилей
WITH limits AS (
    SELECT  
        PERCENTILE_DISC(0.99) WITHIN GROUP (ORDER BY total_area) AS total_area_limit,
        PERCENTILE_DISC(0.99) WITHIN GROUP (ORDER BY rooms) AS rooms_limit,
        PERCENTILE_DISC(0.99) WITHIN GROUP (ORDER BY balcony) AS balcony_limit,
        PERCENTILE_DISC(0.99) WITHIN GROUP (ORDER BY ceiling_height) AS ceiling_height_limit_h,
        PERCENTILE_DISC(0.01) WITHIN GROUP (ORDER BY ceiling_height) AS ceiling_height_limit_l
    FROM flats     
),

-- CTE, определяющий id объявлений, которые не содержат выбросы, оставляющий пропущенные данные
filtered_id AS(
    SELECT id
    FROM flats  
    WHERE 
        total_area < (SELECT total_area_limit FROM limits)
        AND (rooms < (SELECT rooms_limit FROM limits) OR rooms IS NULL)
        AND (balcony < (SELECT balcony_limit FROM limits) OR balcony IS NULL)
        AND ((ceiling_height < (SELECT ceiling_height_limit_h FROM limits)
            AND ceiling_height > (SELECT ceiling_height_limit_l FROM limits)) OR ceiling_height IS NULL)
),

--СТЕ, определяющий для каждого объявления месяц выставления объявления на продажу и месяц его снятия
sezon AS(
	SELECT
		f.id
		, first_day_exposition
		, type
		, f.total_area
--разделение объявлений на категории по месяцу выставления объявления на продажу
		, TO_CHAR(first_day_exposition, 'TMMonth') AS first_month
--выделение номера месяца публикации объявления для дальнейшей сортировки в календарном порядке
		, EXTRACT(MONTH FROM first_day_exposition) AS first_month_num
--разделение объявлений на категории по месяцу снятия объявления с продажи
		, TO_CHAR(first_day_exposition + (days_exposition * INTERVAL '1 day'), 'TMMonth') AS sold_month
--выделение номера месяца снятия с публикации для дальнейшей сортировки в календарном порядке
		, EXTRACT(MONTH FROM first_day_exposition + (days_exposition * INTERVAL '1 day')) AS sold_month_num
--определение стоимости квадратного метра недвижимости 
		, a.last_price::numeric/f.total_area::numeric AS ad_price
--объединение необходимых таблиц по их первичным ключам
	FROM flats AS f
	LEFT JOIN advertisement AS a ON f.id=a.id
	LEFT JOIN type AS t ON f.type_id=t.type_id
--фильтрация по типу населённого пункта 'город' и объявлениям, опубликованными в период с 2015 по 2018 годы, исключающая объявления с аномальными значениями
	WHERE type='город' AND f.id IN (SELECT * FROM filtered_id) AND first_day_exposition BETWEEN '2015-01-01' AND '2018-12-31'
),

--СТЕ, определяющий число опубликованных объявлений, среднюю стоимость кв. метра и среднюю площадь недвижимости, сгруппированных по месяцам публикации
first_statistic AS(
	SELECT
		first_month AS month_ad
--определение числа опубликованных объявлений
		, COUNT(id) AS first_count
--определение средней стоимости квадратного метра недвижимости
		, ROUND(AVG(ad_price),2) AS first_price_avg
--определение средней площади недвижимости
		, ROUND(AVG(total_area)::numeric,2) AS first_total_area_avg
		, first_month_num AS sort
	FROM sezon
	GROUP BY
		first_month
		, first_month_num
),

--СТЕ, определяющий число снятых объявлений, среднюю стоимость кв. метра и среднюю площадь недвижимости, сгруппированных по месяцам снятия объявления 
sold_statistic AS(
	SELECT
		sold_month AS month_ad
--определение числа снятых объявлений
		, COUNT(id) AS sold_count
--определение средней стоимости квадратного метра недвижимости
		, ROUND(AVG(ad_price),2) AS sold_price_avg
--определение средней площади недвижимости
		, ROUND(AVG(total_area)::numeric,2) AS sold_total_area_avg
		, sold_month_num AS sort
	FROM sezon
	GROUP BY
		sold_month
		, sold_month_num
)

-- Основной запрос, объединяющий статистику по месяцам
SELECT
	COALESCE(fstat.month_ad, sstat.month_ad) AS month_advertisement
	, fstat.first_count
	, fstat.first_price_avg
	, fstat.first_total_area_avg
	, sstat.sold_count
	, sstat.sold_price_avg
	, sstat.sold_total_area_avg
FROM first_statistic AS fstat
FULL JOIN sold_statistic AS sstat ON fstat.month_ad=sstat.month_ad
WHERE COALESCE(fstat.month_ad, sstat.month_ad) IS NOT NULL
ORDER BY fstat.sort;

--month_advertisement|first_count|first_price_avg|first_total_area_avg|sold_count|sold_price_avg|sold_total_area_avg|
---------------------+-----------+---------------+--------------------+----------+--------------+-------------------+
--January            |        735|      106106.24|               59.16|      1225|     104947.31|              57.53|
--February           |       1369|      103058.51|               60.10|      1048|     103883.72|              61.12|
--March              |       1119|      102429.95|               60.00|      1071|     106832.40|              60.37|
--April              |       1021|      102632.41|               60.60|      1031|     102444.24|              59.22|
--May                |        891|      102465.12|               59.19|       729|      99724.07|              57.78|
--June               |       1224|      104802.15|               58.37|       771|     101863.69|              59.82|
--July               |       1149|      104488.96|               60.42|      1108|     102290.72|              58.54|
--August             |       1166|      107034.70|               58.99|      1137|     100036.51|              56.83|
--September          |       1341|      107563.12|               61.04|      1238|     104070.07|              57.49|
--October            |       1437|      104065.11|               59.43|      1360|     104317.33|              58.86|
--November           |       1569|      105048.80|               59.58|      1301|     103791.36|              56.71|
--December           |       1024|      104775.39|               58.84|      1175|     105504.52|              59.26|
```

<a id='metka_4'></a>
### Выводы

- В Санкт-Петербурге и Ленинградской области доминируют два сегмента: 31-90 дней (18% и 11%) и 181+ дней (21% и 10%), что говорит о том, что почти пятая часть объявлений в Санкт-Петербурге активна более чем полгода, ещё пятая часть – квартал. В Ленинградской области объявлений «зависает» меньше.
- В Санкт-Петербурге дольше продаётся дорогая и крупная недвижимость, а в Ленинградской области - наоборот. Первые покупатели ориентированы на комфорт, вторые – на доступность цены, поэтому выбор в пользу студий и компактности. Поэтому в Санкт-Петербурге больше площадь – дороже стоимость, в Ленинградской области – наоборот.
- Влияние таких характеристик, как балкон, этаж, высота потолков и т.п. незначительно и поэтому они не являются ключевыми.
- Осень - лучшее время для публикации объявлений, а декабрь – пик продаж, летом рынок замедляется.
- Высокая стартовая цена не гарантирует быструю продажу.

<a id='metka_5'></a>
### Рекомендации

- в Санкт-Петербурге продавцам размещать объявления в октябре-ноябре (есть шансы продать недвижимость уже в декабре), закладывайте на продажу более полугода, если недвижимость крупная.
- в Ленинградской области продавцам избегать завышения цены – это может затянуть сделку, а акцентироваться на бюджетных недвижимостях – они продаются быстрее.
- покупателям активизироваться в 4 квартале (сезон спроса), а летом «притормозить».